# Can AI compound the slow loop of Huntington's Disease research?

**HD Research Hub — Gemma 4 Good Hackathon (Health track)**

Huntington's Disease is a slow loop. A handful of labs, a corpus of papers that grows by a few hundred a year, clinical trials that read out over years, and patients waiting in between. What an AI research-acceleration platform does is shorten that loop — read every new paper the day it lands, extract findings buried in figures, surface drug-repurposing hypotheses against ranked targets, and let researchers (and families) ask grounded, cited questions in plain language.

This is the **only Huntington's Disease entry** in the competition, and — looking at the public code gallery — the **only research-acceleration platform** in a field of end-user assistants, image classifiers, and clinical chatbots. We are not competing with HDSA, HDBuzz, or HDYO. We complement them with an AI/data lens.

## Table of contents

1. Why Gemma 4
2. Setup (Kaggle-attached model with safe local fallback)
3. The compounding knowledge-base moat
4. Multimodal demo — reading figures
5. Function calling demo — agentic chatbot over our corpus
6. The edge story — same code on a Jetson AGX Orin
7. Honest evaluation — tool selection
8. Limitations & future work
9. Gemma 4 usage statement, trademark, license
10. Links


## 1. Why Gemma 4

Gemma 4 is the model HD Research Hub needed:

- **Multimodal vision** — reads paper figures (survival curves, western blots, dose-response charts). The quantitative findings that live only in a chart never made it into the abstract; we extract them and feed them into the same knowledge base the chatbot answers from.
- **Native function calling** — the agentic chatbot doesn't run fixed RAG. Gemma 4 picks tools — `search_papers`, `get_clinical_trials`, `get_target_info`, `get_experiment_findings`, `get_latest_papers` — and we execute them, feed results back, and let the model compose a cited answer.
- **Edge-deployable** — `gemma4:latest` runs in production on a Jetson AGX Orin at home for the daily research pipeline. No cloud bill, no API rate limit, no patient data leaving the box. The hosted Gemma 4 path (AI Studio, `gemma-4-31b-it`) backs the live chatbot because Vercel serverless can't reach the LAN Jetson.
- **Apache 2.0** — open weights, open license. We can ship.

One model, two backends, one codebase (`src/llm.py`). That's the architecture this notebook proves.


## 2. Setup

This notebook is built to be **runnable on Kaggle with no external API key** when the Gemma 4 model is attached via the `Add Input` panel (`google/gemma-4`). It also runs locally — set `HD_LLM_BACKEND=ollama` for a Jetson/local Ollama with `gemma4:latest`, or `HD_LLM_BACKEND=aistudio` with a `GEMINI_API_KEY` for the hosted path.

We try the Kaggle attached model first, then fall back to the `src.llm` backends. Either way the public surface — `ask`, `ask_vision`, `ask_with_tools` — is identical.


In [ ]:
# Lightweight deps. The Kaggle environment already has torch, transformers, matplotlib, PIL.
import sys, subprocess
def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import requests, PIL  # noqa: F401
except ImportError:
    pip("requests", "pillow")


In [ ]:
# Clone the HD Research Hub repo so we can import src/llm.py, src/chat_tools.py
# and read the data/ snapshots. On Kaggle this drops into /kaggle/working.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/jravinder/hd-research-agent.git"
LOCAL_REPO = Path("/Users/red/Documents/github/hd-research-agent")

if LOCAL_REPO.exists():
    REPO = LOCAL_REPO
    print(f"Using local checkout at {REPO}")
else:
    target = Path.cwd() / "hd-research-agent"
    if not target.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(target)])
    REPO = target
    print(f"Cloned repo to {REPO}")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


In [ ]:
# Backend selection. Three paths, tried in order:
#   1. Kaggle attached model (google/gemma-4) via transformers — no API key needed
#   2. Local/Jetson Ollama (HD_LLM_BACKEND=ollama)
#   3. Hosted AI Studio (HD_LLM_BACKEND=aistudio, GEMINI_API_KEY=...)
#
# We expose ONE adapter — ask / ask_vision / ask_with_tools — so the rest of
# the notebook never branches.

import os
from typing import Any

KAGGLE_MODEL_PATH = "/kaggle/input/gemma-4/transformers/gemma-4-9b-it/1"  # adjust if attached differently
_KAGGLE_MODEL = None
_KAGGLE_PROCESSOR = None

def _try_load_kaggle_gemma4():
    global _KAGGLE_MODEL, _KAGGLE_PROCESSOR
    if _KAGGLE_MODEL is not None:
        return True
    # Look for any attached Gemma 4 weights under /kaggle/input
    candidates = []
    root = "/kaggle/input"
    if os.path.isdir(root):
        for dirpath, _, files in os.walk(root):
            if any(f.endswith(".safetensors") or f == "config.json" for f in files):
                if "gemma" in dirpath.lower():
                    candidates.append(dirpath)
    if not candidates:
        return False
    model_dir = candidates[0]
    try:
        import torch
        from transformers import AutoProcessor, AutoModelForImageTextToText
        _KAGGLE_PROCESSOR = AutoProcessor.from_pretrained(model_dir)
        _KAGGLE_MODEL = AutoModelForImageTextToText.from_pretrained(
            model_dir, torch_dtype=torch.bfloat16, device_map="auto"
        )
        print(f"Loaded Kaggle Gemma 4 from {model_dir}")
        return True
    except Exception as e:
        print(f"Kaggle Gemma 4 load failed ({e}); falling back to src.llm backend.")
        return False


def _kaggle_generate(messages, max_new_tokens=512):
    import torch
    inputs = _KAGGLE_PROCESSOR.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(_KAGGLE_MODEL.device)
    with torch.inference_mode():
        out = _KAGGLE_MODEL.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    return _KAGGLE_PROCESSOR.decode(new_tokens, skip_special_tokens=True).strip()


USE_KAGGLE = _try_load_kaggle_gemma4()

if not USE_KAGGLE:
    # Fall through to src.llm — backend is whatever HD_LLM_BACKEND says.
    os.environ.setdefault("HD_LLM_BACKEND", "ollama")
    from src.llm import ask as _srcllm_ask, ask_vision as _srcllm_vision, ask_with_tools as _srcllm_tools
    BACKEND = f"src.llm:{os.environ['HD_LLM_BACKEND']}"
else:
    BACKEND = "kaggle-attached:gemma-4"

print(f"Active backend: {BACKEND}")


In [ ]:
# Adapter — same three entry points whether we're on Kaggle weights, Ollama,
# or AI Studio. Tool calling on the Kaggle path uses a small text-protocol
# (we prompt Gemma 4 to emit a JSON object {tool, args}), parsed below.
import json, re

def ask(prompt: str, system: str = "", temperature: float = 0.3) -> str:
    if USE_KAGGLE:
        msgs = []
        if system:
            msgs.append({"role": "system", "content": [{"type": "text", "text": system}]})
        msgs.append({"role": "user", "content": [{"type": "text", "text": prompt}]})
        return _kaggle_generate(msgs, max_new_tokens=512)
    return _srcllm_ask(prompt, system=system, temperature=temperature)


def ask_vision(prompt: str, images: list, system: str = "", temperature: float = 0.3) -> str:
    if USE_KAGGLE:
        from PIL import Image
        import io, base64
        pil_imgs = []
        for im in images:
            if isinstance(im, (bytes, bytearray)):
                pil_imgs.append(Image.open(io.BytesIO(im)).convert("RGB"))
            elif isinstance(im, str) and os.path.exists(im):
                pil_imgs.append(Image.open(im).convert("RGB"))
            elif isinstance(im, Image.Image):
                pil_imgs.append(im.convert("RGB"))
            else:  # assume base64 string
                pil_imgs.append(Image.open(io.BytesIO(base64.b64decode(im))).convert("RGB"))
        content = [{"type": "image", "image": img} for img in pil_imgs]
        content.append({"type": "text", "text": prompt})
        msgs = []
        if system:
            msgs.append({"role": "system", "content": [{"type": "text", "text": system}]})
        msgs.append({"role": "user", "content": content})
        return _kaggle_generate(msgs, max_new_tokens=512)
    return _srcllm_vision(prompt, images, system=system, temperature=temperature)


_TOOL_PROTOCOL = '''If you need a tool, emit ONE JSON object on its own line:
{"tool": "<tool_name>", "args": {<kwargs>}}
After tool results are returned, you may emit another tool call or write the final answer.
If you are ready to answer the user directly, just write the answer (no JSON).
'''

def _parse_tool_call(text: str):
    # Find the first JSON object that has a "tool" key.
    for m in re.finditer(r"\{[^{}]*\"tool\"[^{}]*\}", text, re.DOTALL):
        try:
            obj = json.loads(m.group(0))
            if "tool" in obj:
                return [{"name": obj["tool"], "args": obj.get("args", {}) or {}}]
        except json.JSONDecodeError:
            continue
    return []


def ask_with_tools(prompt: str, tools: list, system: str = "", temperature: float = 0.3):
    if USE_KAGGLE:
        tool_list = "\n".join(f"  - {t['name']}({', '.join((t['parameters'].get('properties') or {}).keys())}): {t['description']}" for t in tools)
        sys2 = (system or "") + "\n\nTOOLS AVAILABLE:\n" + tool_list + "\n\n" + _TOOL_PROTOCOL
        text = ask(prompt, system=sys2, temperature=temperature)
        calls = _parse_tool_call(text)
        return text, calls
    return _srcllm_tools(prompt, tools, system=system, temperature=temperature)


print("Adapter ready: ask / ask_vision / ask_with_tools")


## 3. The compounding knowledge-base moat

Most HD resources show titles, abstracts, or journalist summaries. **We chat with the actual research, not summaries.**

- The paper scout pulls full text from PubMed Central, chunks it by section (Methods, Results, Discussion), and indexes it.
- Gemma 4's vision pass extracts findings from figures — survival curves, dose-response charts, western blots — and adds them as new chunks tagged `section: figure-N`.
- The autoresearch loop generates drug-repurposing hypotheses against ranked HD targets; those hypotheses become chunks too.
- The agentic chatbot reads from the same chunk store via `search_papers`.

Every run compounds the KB. That's the moat. The chatbot gets smarter as the corpus grows. Nobody else is doing this for HD.

In the cells below: figure → Gemma 4 vision → quantitative findings → chunk → answerable by the chatbot.


## 4. Multimodal demo — reading figures

We synthesize two HD-style research figures with `matplotlib` (so the demo always runs end-to-end with zero network), then ask Gemma 4 to **describe each figure and extract quantitative findings — numbers, p-values, effect sizes, sample sizes**.

In production (Jetson pipeline), the same `ask_vision` call runs against real PMC open-access figure PNGs.


In [ ]:
# Build two synthetic but realistic HD research figures.
# These are NOT real data — they exist so the multimodal demo is reproducible
# on Kaggle with no network. The production pipeline runs the same ask_vision
# call against real PMC figure PNGs.

import io
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def fig_huntingtin_lowering():
    """Synthetic: huntingtin protein levels over time, ASO vs vehicle."""
    weeks = np.array([0, 4, 8, 12, 16, 20, 24])
    vehicle = np.array([100, 102, 99, 101, 98, 100, 103])
    aso_low = np.array([100, 78, 62, 55, 51, 50, 52])
    aso_high = np.array([100, 65, 41, 32, 28, 27, 30])

    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=140)
    ax.errorbar(weeks, vehicle, yerr=4, marker="o", capsize=3, label="Vehicle (n=12)")
    ax.errorbar(weeks, aso_low,  yerr=5, marker="s", capsize=3, label="ASO 50mg (n=12)")
    ax.errorbar(weeks, aso_high, yerr=6, marker="^", capsize=3, label="ASO 100mg (n=12)")
    ax.set_xlabel("Weeks post-dose")
    ax.set_ylabel("CSF mHTT (% baseline)")
    ax.set_title("Figure 2 — Dose-dependent mHTT lowering, p < 0.001 at 24 wk (100mg)")
    ax.axhline(100, color="gray", linestyle=":", linewidth=0.8)
    ax.legend(frameon=False)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    buf = io.BytesIO(); fig.tight_layout(); fig.savefig(buf, format="png"); plt.close(fig)
    return buf.getvalue()

def fig_survival_curve():
    """Synthetic: Kaplan-Meier-style survival curve in R6/2 mouse model."""
    weeks = np.arange(0, 26)
    # Step-down survival functions
    def km(median):
        s = np.ones_like(weeks, dtype=float)
        for i, w in enumerate(weeks):
            s[i] = max(0.0, 1.0 - 0.5 * (1.0 / (1.0 + np.exp(-(w - median) * 0.6))))
        return s
    vehicle = km(14)
    treated = km(20)

    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=140)
    ax.step(weeks, vehicle, where="post", label="Vehicle (n=20), median 14 wk")
    ax.step(weeks, treated, where="post", label="Compound X (n=20), median 20 wk")
    ax.set_xlabel("Age (weeks)")
    ax.set_ylabel("Survival probability")
    ax.set_ylim(0, 1.05)
    ax.set_title("Figure 4 — Survival in R6/2 mice, log-rank p = 0.003, HR 0.42")
    ax.legend(frameon=False, loc="lower left")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    buf = io.BytesIO(); fig.tight_layout(); fig.savefig(buf, format="png"); plt.close(fig)
    return buf.getvalue()

fig1_bytes = fig_huntingtin_lowering()
fig2_bytes = fig_survival_curve()

# Preview them inline so the reader sees what Gemma 4 is looking at.
from PIL import Image
display(Image.open(io.BytesIO(fig1_bytes)))
display(Image.open(io.BytesIO(fig2_bytes)))


In [ ]:
# Vision call — figure 1: huntingtin lowering
VISION_PROMPT = (
    "You are reading a figure from a Huntington's Disease research paper. "
    "Describe what the figure shows in 2-3 sentences, then extract every "
    "quantitative finding you can read: numbers, percentages, p-values, "
    "effect sizes, sample sizes, dose levels. Be precise; if a value is "
    "not legible, say so."
)

print("=" * 72)
print("FIGURE 1 — Huntingtin lowering dose-response")
print("=" * 72)
out1 = ask_vision(VISION_PROMPT, [fig1_bytes])
print(out1)


In [ ]:
# Vision call — figure 2: survival curve
print("=" * 72)
print("FIGURE 2 — Survival curve in R6/2 mouse model")
print("=" * 72)
out2 = ask_vision(VISION_PROMPT, [fig2_bytes])
print(out2)


## 5. Function calling demo — the agentic chatbot

This is the headline feature. The chatbot is **not** fixed RAG. Gemma 4 sees the five tools we expose, picks which to call, we execute, feed results back, and the model composes a cited answer.

We import `src/chat_tools.py` directly — same code that runs in production (`api/chat.py`). The 5 tools:

- `search_papers(query, top_k)` — semantic + keyword fallback over the full-text corpus
- `get_clinical_trials(status, limit)` — live HD trials from ClinicalTrials.gov
- `get_target_info(gene)` — ranked HD targets table
- `get_experiment_findings(n)` — AI-generated drug-repurposing hypotheses
- `get_latest_papers(days, limit)` — most recent papers in the corpus

The `run_agent` helper below mirrors `api/chat.py::run_agentic` — up to 3 tool-call turns, then compose.


In [ ]:
# Import the live production tool registry — NOT a reimplementation.
from src.chat_tools import TOOLS, dispatch

print(f"Loaded {len(TOOLS)} tools:")
for t in TOOLS:
    print(f"  - {t['name']}: {t['description'][:90]}...")


In [ ]:
# A compact run_agent that mirrors api/chat.py::run_agentic.
# Up to MAX_TURNS tool rounds, then we ask Gemma 4 to compose the final answer.

SYSTEM_PROMPT = """You are the HD Research Hub assistant — an AI that helps people explore Huntington's Disease RESEARCH. You are NOT a doctor.

RULES:
- Prefer calling a tool over guessing.
- Cite PubMed IDs inline using [PMID] when you use a paper.
- End research answers with: *This is AI-generated research analysis, not medical advice.*
- AI-generated hypotheses are UNVALIDATED computational ideas.
"""

MAX_TURNS = 3

def _format_tool_result(name, args, result):
    # Trim long string fields so the model doesn't drown.
    import json as _j
    s = _j.dumps(result, default=str)
    if len(s) > 2400:
        s = s[:2400] + "... (truncated)"
    return f"TOOL: {name}({args})\nRESULT: {s}"


def run_agent(question: str, verbose: bool = True) -> dict:
    tools_used = []
    tool_results = []

    text, calls = ask_with_tools(question, TOOLS, system=SYSTEM_PROMPT)

    turns = 0
    while calls and turns < MAX_TURNS:
        turns += 1
        if verbose:
            print(f"  Turn {turns}: model wants {[c['name'] for c in calls]}")
        round_results = []
        for c in calls:
            name = c.get("name", "")
            args = c.get("args") or {}
            res = dispatch(name, args)
            tools_used.append(name)
            round_results.append({"name": name, "args": args, "result": res})
            tool_results.append({"name": name, "args": args, "result": res})

        block = "\n\n".join(_format_tool_result(r["name"], r["args"], r["result"])
                              for r in round_results)
        follow = (
            f"USER QUESTION: {question}\n\n"
            f"Tool results:\n\n{block}\n\n"
            "If you need more tools, emit another call. Otherwise, write the final cited answer."
        )
        text, calls = ask_with_tools(follow, TOOLS, system=SYSTEM_PROMPT)

    if not text and tool_results:
        compose = (
            f"USER QUESTION: {question}\n\nTool results:\n"
            + "\n\n".join(_format_tool_result(r["name"], r["args"], r["result"])
                            for r in tool_results)
            + "\n\nWrite the final answer now, citing [PMID] where applicable."
        )
        text = ask(compose, system=SYSTEM_PROMPT)

    return {"question": question, "answer": text, "tools_used": tools_used,
            "tool_results": tool_results}

print("run_agent ready.")


In [ ]:
# Q1 — clinical trials lookup
result1 = run_agent("How many HD trials are currently recruiting?")
print("\n--- ANSWER ---")
print(result1["answer"])
print("\nTools called:", result1["tools_used"])


In [ ]:
# Q2 — top drug-repurposing hypothesis
result2 = run_agent("What's the top drug-repurposing hypothesis from our experiments and what target does it act on?")
print("\n--- ANSWER ---")
print(result2["answer"])
print("\nTools called:", result2["tools_used"])


In [ ]:
# Q3 — corpus search with citations
result3 = run_agent("Search the corpus for ASO/huntingtin-lowering evidence and summarize with PMID citations.")
print("\n--- ANSWER ---")
print(result3["answer"])
print("\nTools called:", result3["tools_used"])


## 6. The edge story

The same `src/llm.py` you just used runs on a **Jetson AGX Orin** on a desk at home for the daily research pipeline:

- 8am UTC cron pulls new PubMed papers
- `paper_analyzer.py` runs Gemma 4 over each paper's text and figures (`ask` + `ask_vision`)
- `repurposing_scanner.py` generates drug hypotheses against the 16 ranked HD targets
- New knowledge-base chunks land in `data/corpus.json` and are searchable by the chatbot the next morning

Zero cloud bill, zero per-token cost, zero data leaving the box. That's what makes overnight autoresearch loops feasible.

**The honest wrinkle:** Vercel serverless can't reach a LAN Jetson. So the **live chatbot** runs Gemma 4 hosted via Google AI Studio (`gemma-4-31b-it`) — the same code, different backend, selected by `HD_LLM_BACKEND`. One codebase, two homes. Edge for the pipeline, hosted for the interactive product. That's the truthful inference topology, and it's what makes Gemma 4 — multimodal + agentic + Apache 2.0 + sized for both edge and hosted — the right model for this project.


## 7. Honest evaluation — does Gemma 4 pick the right tool?

A small 5-row tool-selection sanity check. For each question, what tool *should* Gemma 4 call? What did it actually call? We score by hand. No hype — if some are wrong, we say so.


In [ ]:
# Tool selection eval — 5 simple research questions.
# expected_tool is the FIRST tool we'd expect a reasonable agent to call.

EVAL = [
    ("How many recruiting HD trials are there right now?",                "get_clinical_trials"),
    ("Tell me about the HTT gene as a target.",                            "get_target_info"),
    ("What are our top AI-generated drug repurposing hypotheses?",         "get_experiment_findings"),
    ("What's the latest HD paper this month?",                             "get_latest_papers"),
    ("Find papers discussing somatic CAG instability in striatal neurons.","search_papers"),
]

rows = []
for q, expected in EVAL:
    _, calls = ask_with_tools(q, TOOLS, system=SYSTEM_PROMPT)
    actual = calls[0]["name"] if calls else "(none)"
    correct = (actual == expected)
    rows.append((q, expected, actual, correct))

print(f"{'Question':<60} {'Expected':<26} {'Actual':<26} {'OK'}")
print("-" * 120)
for q, exp, act, ok in rows:
    qd = (q[:57] + "...") if len(q) > 60 else q
    print(f"{qd:<60} {exp:<26} {act:<26} {'YES' if ok else 'NO'}")

correct_n = sum(1 for *_, ok in rows if ok)
print(f"\nTool-selection accuracy: {correct_n}/{len(rows)} = {100*correct_n/len(rows):.0f}%")

# Honesty: if a row is wrong, log why we think it's wrong. Common failure modes
# we've seen: model calls search_papers for everything (over-general), or calls
# nothing at all when the question is direct ("just answer me").
if correct_n < len(rows):
    print("\nNotes on misses (be honest, don't paper over):")
    for q, exp, act, ok in rows:
        if not ok:
            print(f"  - '{q[:60]}' -> picked {act!r} instead of {exp!r}.")
            if act == "search_papers":
                print("    Failure mode: defaulted to general search instead of the specific data tool.")
            elif act == "(none)":
                print("    Failure mode: model answered from prior knowledge without invoking a tool.")


## 8. Limitations & future work

We are data scientists, not doctors. Honest limits:

- **Open-access papers only.** The KB is built from PubMed Central OA. Paywalled work is not indexed. We link out to the source either way.
- **8K context on Ollama (Jetson edge).** Long full-text papers get chunked. Cross-section reasoning is bounded by how we chunk and re-feed.
- **Molecular structure reading is unproven and skipped from this submission.** We considered asking Gemma 4 to read PubChem 2D structure PNGs for blood-brain-barrier-relevant properties; in early testing it was confidently wrong about functional groups, so we cut it rather than ship a broken feature. Honest "we tried, here's what didn't work" beats a half-working demo.
- **Not clinical, on purpose.** A hard medical-redirect guardrail wraps every chat turn. Personal medical questions go to HDSA (`hdsa.org`, `1-800-345-HDSA`), a neurologist, or a genetic counselor. We don't recommend, diagnose, or prognose — by policy.
- **Hypothesis tracker is computational, not validated.** Every AI-generated drug-repurposing idea on the site is labeled UNVALIDATED. It's a starting point for human experts to triage, not a treatment.
- **Tool-selection isn't perfect** (see eval above). When Gemma 4 misses, it typically defaults to `search_papers` for everything. We could harden prompts or add a tiny router model — open work.

Future:
- Real-time PMC figure ingestion in the daily pipeline (Tier 2 in the spec) is wired but underdriven; expand it.
- Cross-paper synthesis — ask "what do these 5 papers collectively say about somatic CAG repeat instability?" with multi-doc reduce.
- A community-suggested hypothesis tracker so HDSA / HDBuzz readers can flag promising AI hypotheses for expert review.


## 9. Gemma 4 usage statement, trademark, license

**How Gemma 4 is used in this submission:**

- `ask()` — text generation across the daily research pipeline, the autoresearch loop, and the live chatbot.
- `ask_vision()` — multimodal figure understanding (this notebook, the production paper analyzer, and the chatbot's image-upload path).
- `ask_with_tools()` — native function calling for the agentic chatbot (5 tools over our shipped data snapshots).

The same `src/llm.py` module backs all three surfaces. Backend is `gemma4:latest` on Jetson Ollama for the offline pipeline, and `gemma-4-31b-it` on Google AI Studio for the live serverless chatbot. Both serve the same Gemma 4 family. No fine-tuning.

**Gemma is a trademark of Google LLC.**

**License:** This notebook and the HD Research Hub repository are released under the **Apache 2.0 license**, matching the Gemma 4 model license.


## 10. Links

- **Repo:** https://github.com/jravinder/hd-research-agent
- **Live demo:** https://hd-research-agent.vercel.app
- **Write-up:** [`docs/HACKATHON.md`](https://github.com/jravinder/hd-research-agent/blob/main/docs/HACKATHON.md)
- **Spec:** [`docs/superpowers/specs/2026-05-14-gemma4-hackathon-design.md`](https://github.com/jravinder/hd-research-agent/blob/main/docs/superpowers/specs/2026-05-14-gemma4-hackathon-design.md)
- **Plan:** [`docs/superpowers/plans/2026-05-15-gemma4-hackathon.md`](https://github.com/jravinder/hd-research-agent/blob/main/docs/superpowers/plans/2026-05-15-gemma4-hackathon.md)

Not affiliated with HDSA, HDBuzz, or HDYO. Educational only. Not medical advice.
